# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
import pandas as pd
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder, FunctionTransformer
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import cross_validate, train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, r2_score

# from sklearn.tree import DecisionTreeClassifier
# from sklearn.metrics import accuracy_score, log_loss, cohen_kappa_score, f1_score

In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt_tot = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
#fires_dt_tot.info()


# Get X and Y

Create the features data frame and target data.

In [3]:
# Create target data frame
df_area_burnt_tot = fires_dt_tot[['area']]
# Create the features data frame
fires_feat_tot = fires_dt_tot.drop(columns=['area'])

In [4]:
#df_area_burnt_tot.describe()

In [5]:
#fires_feat_tot.describe()

Stratified Split Train/Test

In [6]:
# Setting a random seed for GridSearchCV and cross_validate later used, even though we finally found how to embed random_state inside those objects, I left it in case I will need it
random.seed(42)

# Assuming `fires_feat` is your feature set and `df_area_burnt` is your target
stratify_labels = (df_area_burnt_tot == 0).astype(int)  # Convert target into binary labels

fires_feat_train, fires_feat_test, df_area_burnt_train, df_area_burnt_test = train_test_split(
    fires_feat_tot, df_area_burnt_tot, test_size=0.2, stratify=stratify_labels, random_state=42
)

#print(f"Train target distribution: {df_area_burnt_train.value_counts(normalize=True)}")
#print(f"Test target distribution: {df_area_burnt_test.value_counts(normalize=True)}")

# to shorten variable names
fires_feat = fires_feat_train.copy()
df_area_burnt = df_area_burnt_train.copy()
fires_dt = pd.concat([fires_feat,df_area_burnt], axis=1)

Non-Stratified Split Train/Test. Disabled

In [7]:
# # Setting a random seed for GridSearchCV and cross_validate later used, even though we finally found how to embed random_state inside those objects
# random.seed(42)
# # Splitting Strategy: Create training and test datasets
# fires_feat_train, fires_feat_test, df_area_burnt_train, df_area_burnt_test = train_test_split(fires_feat, df_area_burnt, test_size=0.2, random_state=42)
# #print(fires_feat_train.shape, fires_feat_test.shape, df_area_burnt_train.shape, df_area_burnt_test.shape)
# # to shorten variable names
# fires_feat = fires_feat_train.copy()
# df_area_burnt = df_area_burnt_train.copy()
# fires_dt = pd.concat([fires_feat,df_area_burnt], axis=1)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [8]:
# Let's set objects for next section

X1_train = fires_feat_train.copy()
Y1_train = df_area_burnt_train.copy()
#print(type(X1_train),X1_train.shape, X1_train.columns)
#print(type(Y1_train),Y1_train.shape, Y1_train.columns)

Let's create the transformer that will use the next section

In [9]:
# Build transformer
transformer1= ColumnTransformer(
    transformers=[
        ('numeric_transfomer', StandardScaler(), ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain'] ),
        ('onehot', OneHotEncoder(drop = 'first', handle_unknown='infrequent_if_exist'), ['coord_x', 'coord_y', 'month', 'day']), 
    ], remainder='drop'
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [10]:
# Create a binary target: is_not_zero = 1 if y != 0 else 0
y_train_binary = (Y1_train != 0).astype(int).values.ravel()  # This will be your classification target

# Classification pipeline
pipe_classification = Pipeline([
    ('preprocessing', transformer1),
    ('model', LogisticRegression())
])


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

Pipeline E

In [ ]:
param_grid = {
    'model__solver': ['liblinear'],#['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],  #['newton-cg'],#
    'model__C': [0.01],#[0.01, 0.1, 1, 10, 100, 1000],  # Regularization strength  [0.01],#
    'model__penalty': ['l2']#['l1', 'l2','elasticnet']  # Some only work with some solvers, it fails but does not give error, still checks all   ['l2']#
}
# Try:
#0.10	l1	saga	
#0.01	l2	liblinear
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)        # 5-fold cross-validation
scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'f1': 'f1',
    'recall': 'recall'
}
grid_search = GridSearchCV(
    estimator=pipe_classification,
    param_grid=param_grid,
    scoring = scoring,
    refit='f1',      # or 'accuracy', 'f1', etc.
    cv=cv_strategy,           
    n_jobs=-1               # use all cores
)

grid_search.fit(X1_train, y_train_binary)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('numeric_transfomer',
                                                                         StandardScaler(),
                                                                         ['ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('onehot',
                                                                         OneHotEncoder(drop='first',
                                                                                       handle_unknown='infrequent_if_exist'),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'month',
                                                                          'day'])])),
                                       ('model', LogisticRegression())]),
             n_jobs=-1,
             param_grid={'model__C': [0.01], 'model__penalty': ['l2'],
                         'model__solver': ['liblinear']},
             refit='f1',
             scoring={'accuracy': 'accuracy', 'f1': 'f1', 'recall': 'recall',
                      'roc_auc': 'roc_auc'})

In [12]:
print("Best score (CV average):", grid_search.best_score_)


Best score (CV average): 0.6471981012248244


In [13]:
print("Best parameters:", grid_search.best_params_)


Best parameters: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'liblinear'}


In [14]:
best_model_classification = grid_search.best_estimator_


In [15]:
import pandas as pd

cv_results_df = pd.DataFrame(grid_search.cv_results_)
#cv_results_df.sort_values(by='mean_test_score', ascending=False)#.head()
a=cv_results_df[['param_model__C', 'param_model__penalty', 'param_model__solver', 'mean_test_roc_auc', 'mean_test_accuracy', 'mean_test_f1', 'mean_test_recall', 'rank_test_roc_auc', 'rank_test_accuracy', 'rank_test_f1','rank_test_recall']]
a.sort_values(by='rank_test_f1', ascending=True).head()

,param_model__C,param_model__penalty,param_model__solver,mean_test_roc_auc,mean_test_accuracy,mean_test_f1,mean_test_recall,rank_test_roc_auc,rank_test_accuracy,rank_test_f1,rank_test_recall
0,0.01,l2,liblinear,0.533881,0.559271,0.647198,0.782424,1,1,1,1


In [16]:
y_class_pred  = best_model_classification.predict(X1_train)
#print(y_class_pred.shape)
# Count how many were predicted as each class
import numpy as np
unique, counts = np.unique(y_class_pred, return_counts=True)
#print(dict(zip(unique, counts)))  # e.g., {0: 480, 1: 520}  # {0: 89, 1: 324}


In [17]:
# y_proba_train = best_model.predict_proba(X1_train)[:, 1]  # Probability of class 1 (non-zero)
# threshold = 0.6
# y_pred_custom = (y_proba_train > threshold).astype(int)
# y_pred_custom


## Model Pipeline, another one

In [18]:
# Pipeline B = preproc1 + baseline
pipe_regression = Pipeline([
    ('preprocessing', transformer1),
    ('poly', PolynomialFeatures()),
    ('model', LinearRegression())
])

In [19]:
# Subset for regression: non-zero targets only
mask_non_zero  = y_class_pred == 1
X_non_zero = X1_train.loc[mask_non_zero].copy()
y_non_zero = Y1_train.loc[mask_non_zero].copy()
# Fit the pipeline
#pipe_regression.fit(X_non_zero, y_non_zero)
# print(mask_non_zero.shape)
# print(X_non_zero.shape)
# print(X1_train.shape)
# print(X1_train)
# print(X_non_zero)

# Tune Hyperparams, another one

In [20]:
param_grid = {
    'poly__degree': [1, 2, 3, 4],
    'poly__interaction_only': [False, True],
    'poly__include_bias': [False],
    'model__fit_intercept': [True, False]
}
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'neg_rmse': 'neg_root_mean_squared_error',
    'r2': 'r2'
}
grid_search = GridSearchCV(pipe_regression, param_grid, scoring=scoring, refit='neg_rmse', cv=cv_strategy, return_train_score = True) 
grid_search.fit(X_non_zero, y_non_zero)

print("Best parameters:", grid_search.best_params_)
# refit='neg_mse'  # Best parameters: {'model__fit_intercept': False, 'poly__degree': 1, 'poly__include_bias': False, 'poly__interaction_only': False}

c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn

Best parameters: {'model__fit_intercept': False, 'poly__degree': 1, 'poly__include_bias': False, 'poly__interaction_only': False}


c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [21]:
# Get the best pipeline from GridSearchCV, which is the fit to the whole trainiing set with the best grid parameters
best_pipeline_regression = grid_search.best_estimator_
# Predict regression values
y_reg_pred = best_pipeline_regression.predict(X_non_zero)
#print(y_reg_pred.shape)

In [22]:
#print(Y1_train.shape)
# Step 3: Initialize full prediction array
y_full_pred = np.zeros_like(Y1_train)
# Step 4: Insert regression predictions into predicted non-zero locations
y_full_pred[mask_non_zero] = y_reg_pred
#print(mask_non_zero.shape)
#print(y_reg_pred.shape)
#print(y_full_pred.shape)


In [23]:
# Step 5: Evaluate training dataset
rmse = root_mean_squared_error(Y1_train, y_full_pred)
r2 = r2_score(Y1_train, y_full_pred)

print(f"Combined RMSE: {rmse:.2f}")     # Combined RMSE: 65.51
print(f"Combined R²: {r2:.2f}")         # Combined R²: 0.08

Combined RMSE: 65.51
Combined R²: 0.08


In [ ]:
# Step 5: Evaluate TEST dataset
X1_test = fires_feat_test
Y1_test = df_area_burnt_test

# Classification Estimator
y_class_pred_TEST  = best_model_classification.predict(X1_test)

# Subset for regression: non-zero targets only
mask_non_zero_TEST  = y_class_pred_TEST == 1
X_non_zero_TEST = X1_test.loc[mask_non_zero_TEST].copy()
y_non_zero_TEST = Y1_test.loc[mask_non_zero_TEST].copy()

# Predict regression values
y_reg_pred_TEST = best_pipeline_regression.predict(X_non_zero_TEST)


# Step 3: Initialize full prediction array
y_full_pred_test = np.zeros_like(Y1_test)
# Step 4: Insert regression predictions into predicted non-zero locations
y_full_pred_test[mask_non_zero_TEST] = y_reg_pred_TEST


rmse = root_mean_squared_error(Y1_test, y_full_pred_test)
r2 = r2_score(Y1_test, y_full_pred_test)

print(f"Combined RMSE: {rmse:.2f}")     # Combined RMSE: 41.88
print(f"Combined R²: {r2:.2f}")         # Combined R²: -0.16

Combined RMSE: 41.88
Combined R²: -0.16


c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


REGRESSION ONLY STUFF, BELOW. Used and archived

In [25]:
# # Get the best pipeline from GridSearchCV, which is the fit to the whole trainiing set with the best grid parameters
# best_pipeline_regression = grid_search.best_estimator_

# # Access the regression step and grab coefficients
# coefficients = best_pipeline_regression.named_steps['model'].coef_
# intercept = best_pipeline_regression.named_steps['model'].intercept_

# print("Coefficients:", coefficients)
# print("Number of coefficients:", len(coefficients[0])) # Number of coefficients: 20
# print("Intercept:", intercept) # Intercept: 0.0

In [26]:
# # Get the transformed feature names from the preprocessing step
# preprocessor = best_pipeline_regression.named_steps['preprocessing']
# preprocessed_feature_names = preprocessor.get_feature_names_out()

# # Now get polynomial feature names based on these
# poly = best_pipeline_regression.named_steps['poly']
# feature_names = poly.get_feature_names_out(input_features=preprocessed_feature_names)

# # Combine with coefficients
# coefs = best_pipeline_regression.named_steps['model'].coef_.ravel() # .ravel() flattens from (1,27) or (27,1) to (27,)
# coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})

# print(coef_df)

In [27]:
# # Get the cross-validation results as a DataFrame
# cv_results = grid_search.cv_results_

# # Extract the mean negative MSE and r2 scores
# neg_rmse_scores = cv_results['mean_test_neg_rmse']

# # Convert to RMSE
# rmse_scores = (-neg_rmse_scores)

# # And if you're curious about the best RMSE, which is the average RMSE of the folds:
# best_rmse = -grid_search.best_score_  # grid_search.best_score_ is already neg_root_mean_squared_error, we don't need to do sqrt

# # View all RMSE scores across the grid
# for params, score in zip(cv_results['params'], rmse_scores):
#     print(f"Params: {params} → RMSE: {score:.2f}")
# print()  
# print("Best RMSE from CV:", best_rmse)          # Best RMSE from CV: 57.86502073548819
# print()
# print((cv_results.keys()))

In [28]:
# r2_scores = cv_results['mean_test_r2']
# for params, score in zip(cv_results['params'], r2_scores):
#     print(f"Params: {params} → R²: {score:.2f}")
# print()  

# # Get the cross-validation results
# cv_results = grid_search.cv_results_

# # Extract the mean test R² scores
# r2_scores = cv_results['mean_test_r2']

# # Find the best RMSE model index
# best_index = grid_search.best_index_  # Index of best model (based on neg_rmse)

# # Get the corresponding R² score for that best model
# best_r2 = r2_scores[best_index]

# print("Best R² from CV:", best_r2)  # Best R² from CV: -2.442638990981698

In [29]:
# #After Cross Validation process inside GridSearch and after assessing at the errors, if the polynomial coeff is 1, then we fit a standard LinearRegression to the whole Trainig set without folding subsets, and take those coeff.
# # Get the negative rrot mean squared error and R2 obtained after retrained on the entire Training dataset

# # Make predictions on the full training data
# train_preds_1 = best_pipeline.predict(X1_train)

# # Compute RMSE root mean squared error
# train_rmse_1 = root_mean_squared_error(Y1_train, train_preds_1)

# print("RMSE on full training data:", train_rmse_1)  # RMSE in original target units     # RMSE on full training data: 66.48140554239002

# # Compute R² on the full training set
# train_r2_1 = r2_score(Y1_train, train_preds_1)

# print("R² score on full training data:", train_r2_1)        # R² score on full training data: 0.05571835082538645

In [30]:
# # Get the negative root mean squared error RMSE and R2 on the TEST dataset
# # Recover TEST dataset
# X1_test = fires_feat_test.copy()
# Y1_test = df_area_burnt_test.copy()

# # Get the best estimator from GridSearchCV
# best_model = grid_search.best_estimator_

# # Predict on the training or test set
# Y1_pred = best_model.predict(X1_test)

# # Calculate RMSE
# rmse = root_mean_squared_error(Y1_test, Y1_pred)

# # Calculate R-squared
# r2 = r2_score(Y1_test, Y1_pred)

# print("RMSE:", rmse)    # RMSE: 42.07356539407348
# print("R²:", r2)        # R²: -0.16687246410040668

# Evaluate

+ Which model has the best performance?

# Export

+ Save the best performing model to a pickle file.

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.